# 데이터 전처리

> ZIP에 포함된 원본 분석 노트북을 저장소 구조에 맞게 정리한 버전입니다.
> 저장소 공개를 위해 개인 PC의 절대경로와 인증정보는 상대경로·환경변수 방식으로 정리했습니다.


In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
# 데이터 로드

df = pd.read_csv("../data/raw/train.csv")
print("Loaded:", df.shape)
df.head()

In [ ]:
df_raw=df.copy()

In [ ]:
print(df.info())
print(df.describe())

In [ ]:
print(df.isnull().sum())

In [ ]:

# 기본 점검

print("\n[Columns]\n", df.columns.tolist())
print("\n[dtypes]\n", df.dtypes)

print("\n[Duplicated rows]", df.duplicated().sum())
print("[ID unique / total]", df["ID"].nunique(), "/", len(df))
print("[ID duplicated]", df["ID"].duplicated().sum())
print("[(User-ID, Book-ID) duplicated]", df.duplicated(subset=["User-ID", "Book-ID"]).sum())

na_counts = df.isna().sum().sort_values(ascending=False)
print("\n[NA counts > 0]\n", na_counts[na_counts > 0])

print("\n[Book-Rating distribution]\n", df["Book-Rating"].value_counts(dropna=False).sort_index())

In [ ]:

# 텍스트 정규화 함수

NA_TOKENS = {"", "n/a", "na", "none", "null", "unknown"}

def normalize_text_min(x):

    if pd.isna(x):
        return np.nan 
    s = str(x).strip().lower() # 앞뒤 공백 제거(strip), 소문자 통일(lower)
    s = re.sub(r"\s+", " ", s) #연속 공백(2칸 이상)을 1칸으로 축약
    if s in NA_TOKENS:
        return np.nan # 결측 토큰이면 np.nan 반환
    return s

In [ ]:

# 텍스트 컬럼 정규화 적용

for col in ["Location", "Book-Title", "Book-Author", "Publisher"]:
    df[col] = df[col].apply(normalize_text_min)

In [ ]:

# 작가/출판사 결측 행 삭제

before = len(df)
df = df.dropna(subset=["Book-Author", "Publisher"]).copy()  # 특정 컬럼 NaN인 행 삭제
print(f"\n[Drop rows where Book-Author or Publisher missing] {before} -> {len(df)}")

In [ ]:

# Location_country 생성 (마지막 쉼표 뒤만) + 결측 행 삭제

df["Location_country"] = (
    df["Location"].astype(str)
      .str.strip().str.lower()
      .str.replace(r"\s+", " ", regex=True)
      .str.rsplit(",", n=1).str[-1]   # 마지막 쉼표 기준 오른쪽만 추출
      .str.strip()
)

# 결측 토큰이면 NaN으로 바꾸고 행 삭제
df.loc[df["Location_country"].isin(NA_TOKENS), "Location_country"] = np.nan
before = len(df)
df = df.dropna(subset=["Location_country"]).copy() 
print(f"[Drop rows where Location_country missing] {before} -> {len(df)}")

In [ ]:

# Year-Of-Publication 처리 + 결측 행 삭제

df["Year-Of-Publication"] = pd.to_numeric(df["Year-Of-Publication"], errors="coerce")
df.loc[df["Year-Of-Publication"] == -1, "Year-Of-Publication"] = np.nan
print("\n[YOP NA count after -1->NaN]:", df["Year-Of-Publication"].isna().sum())

before = len(df)
df = df.dropna(subset=["Year-Of-Publication"]).copy() 
print(f"[Drop rows where Year-Of-Publication missing] {before} -> {len(df)}")

print("\nDone (preprocessing before rating filtering / p-core). Current df shape:", df.shape)

In [ ]:

#중간 점검, 국가 상위 빈도 확인

print("\nTop 30 Location_country:")
print(df["Location_country"].value_counts().head(30))

In [ ]:

# Book-Rating에서 0 제거 후 1~10만 회귀 대상으로 사용


df_explicit = df[df["Book-Rating"].between(1, 10)].copy()
print("\n[df_explicit] shape:", df_explicit.shape)
print("[df_explicit] rating distribution:\n", df_explicit["Book-Rating"].value_counts().sort_index())

# 유저/책별 상호작용 개수 분포

user_cnt_pre = df_explicit.groupby("User-ID").size()
item_cnt_pre = df_explicit.groupby("Book-ID").size()
print("\n[Pre k-core] user rating count describe:\n", user_cnt_pre.describe())
print("\n[Pre k-core] book rating count describe:\n", item_cnt_pre.describe())

In [ ]:

# (2) p-core filtering (min_user=3, min_item=3) -> df_final

def p_core_filter(df_in, user_col="User-ID", item_col="Book-ID", min_user=3, min_item=3, max_iter=10):
    out = df_in.copy()
    for _ in range(max_iter):
        before = len(out)

        # 유저 기준: min_user 미만 제거
        user_cnt = out[user_col].value_counts()
        out = out[out[user_col].isin(user_cnt[user_cnt >= min_user].index)]

        # 아이템 기준: min_item 미만 제거
        item_cnt = out[item_col].value_counts()
        out = out[out[item_col].isin(item_cnt[item_cnt >= min_item].index)]

        if len(out) == before:
            break
    return out

df_final = p_core_filter(df_explicit, min_user=3, min_item=3)
print("\n[df_final after p-core (3,3)] shape:", df_final.shape)

# p-core가 제대로 적용됐는지(최소가 3 이상인지) 확인
print("min ratings per user:", df_final.groupby("User-ID").size().min())
print("min ratings per book:", df_final.groupby("Book-ID").size().min())

# (선택) (User-ID, Book-ID) 중복 재확인
print("(User-ID, Book-ID) duplicated:", df_final.duplicated(subset=["User-ID", "Book-ID"]).sum())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid")

def run_eda(d, name="df"):
    print(f"\n==================== {name} ====================")
    print("shape:", d.shape)
    print("n_users:", d["User-ID"].nunique(), "n_books:", d["Book-ID"].nunique())

    # 1) Book-Rating 분포 (가장 중요)
    plt.figure(figsize=(8, 4))
    order = sorted(d["Book-Rating"].dropna().unique())
    sns.countplot(data=d, x="Book-Rating", order=order)
    plt.title(f"{name} | Book-Rating Distribution")
    plt.tight_layout()
    plt.show()

    user_cnt = d.groupby("User-ID").size()
    plt.figure(figsize=(7, 4))
    user_cnt.plot(kind="hist", bins=50, logy=True, edgecolor="black")
    plt.title(f"{name} | #Ratings per User (log y)")
    plt.xlabel("#ratings by user")
    plt.ylabel("#users (log scale)")
    plt.tight_layout()
    plt.show()

    item_cnt = d.groupby("Book-ID").size()
    plt.figure(figsize=(7, 4))
    item_cnt.plot(kind="hist", bins=50, logy=True, edgecolor="black") 
    plt.title(f"{name} | #Ratings per Book (log y)")
    plt.xlabel("#ratings for a book")
    plt.ylabel("#books (log scale)")
    plt.tight_layout()
    plt.show()

run_eda(df_raw, "raw df (0 included)")
run_eda(df_final, "df_final (p-core 3,3)")
